Tematem poniższego labolatorium jest estymacja parametrów dla wybranej populacji.

Do tej pory kiedy mieliśmy do czynienia z parametrami zmiennych losowych takimi jak wartość oczekiwana, czy wariancja, znaliśmy dokładnie własności rozkładu zmiennej losowej dla której parametry mamy wyznaczyć. W statystyce jednak już takiego luksusu nie mamy, ponieważ świat został nam dany bez dokumentacji i nie wiemy czy, na przykład, wzrost ludzi w Krakowie jest losowany z rozkładu wykładniczego o parametrze 30, rozkładu normalnego ze średnią 160 i wariancją 20, czy też zwyczajnie z rozkładu jednostajnego z przedziału [100, 250]. Ale to nie jest bardzo poważny problem, ponieważ często tak dokładna wiedza nie jest nam potrzebna. Istotnie, wystarczającą informacją zdaje się nam na codzień średnia długość życia, czy też średnia zdawalność egzaminu z prawdopodobieństwa. I tu z pomocą przychodzą nam statystyki zwane estymatorami.

Wyróżniamy dwa typy estymacji: punktową i przedziałową. Zaczniemy od punktowej. W tym przypadku próbujemy wyznaczyć dokładny parametr populacji o nieznanym rozkładzie na podstawie próby z niej pobranej (próba to jest zestaw wyników jaki otrzymamy losując wartości jednorodnie z populacji). Na podstawie wybranej statystyki (statystyka jest to po prostu dowolna funkcja, która dostaje wyniki z próby i wypluwa wartość rzeczywistą) dokonujemy obliczeń i mamy estymację. Oczywiście łatwo zauważyć pierwszy problem: jeżeli statystyka jest dowolną funkcją to dlaczego miałaby mieć coś wspólnego z szukanym parametrem? Tu właśnie do gry wkraczają twierdzenia graniczne i inne metody probabilistyczne, które pomagają nam ustalić które statystyki są dobrymi estymatorami, a które nie. I tak, dla przykładu, okazuje się że najlepszym estymatorem wartości średniej populacji jest zwyczajne wzięcie wartości średniej z całej próbki. Zatem jeżeli chcilibyśmy zasugerować sensowną średnią wzrostu mieszkańców Krakowa wystarczy zapytać stu losowych mieszkańców miasta (fragment z losowym dobieraniem losowych mieszkańców jest dużo bardziej podchwytliwy niż się wydaje i nie należy o tym zapominać) wziąć średnią z odpowiedzi i mamy estymację punktową wzrostu mieszkańców Krakowa.

Sprawa nie jest tak prosta z estymacją punktową wariancji. Wydawać by się mogło, że wystarczy postąpić podobnie jak przy wartości średniej i policzyć $$ \frac{1}{n} \sum_{i=1}^n (X_i- \bar{X})^2,$$ gdzie $X_i$ to wartości z próbek, $n$ liczba elementów w próbce, zaś $\bar{X}$ to średnia z próbki, jednak taki estymator będzie miał wartość oczekiwaną odchyloną od wartości wariancji całej populacji, czyli będzie tak zwanym estymatorem obciążonym. Żeby go poprawić trzeba zastosować formułę $$\frac{1}{n-1} \sum_{i=1}^n (X_i- \bar{X})^2$$ i pierwszym zadaniem jest napisać funkcję która taki nieobciążony estymator wariancji (w skrócie NEW) liczy.

In [31]:
import math
from scipy.stats import norm
from scipy.stats import t
from scipy.stats import chi2

def NEW(lista):  #lista zawierająca wyniki z próby
  n = len(lista)
  x_bar = sum(lista) / len(lista)
  sq_diff = map(lambda x: (x - x_bar)**2, lista)
  return sum(sq_diff) / (n - 1)

#dla przykładu NEW([1,2,3,4,5,6,7,8,9]) powinien zwracać 7.5
print('result:', NEW([1,2,3,4,5,6,7,8,9]))
print('answer:', 7.5)

result: 7.5
answer: 7.5


Jeżeli ktoś zauważył poważny problem z estymacją punktową to znaczy, że dobrze przyswoił materiał związany z prawdopodobieństwem. Jeżeli nie, to teraz warto chwilę się zastanowić co powinno nam w tym pomyśle przeszkadzać.

Problemem tym oczywiście jest prawdopodobieństwo z jakim podana przez nas estymacja jest poprawna. W większości przypadków zmienne losowe służące do badania statystyk są ciągłe, więc prawdopodobieństwo trafienia w dokładną wartość wynosi 0. Co nie znaczy, że estymacja punktowa jest bezwartościowa, często można ją wykorzystywać do ogólnego opisu populacji, a także stosować w innych statystykach gdzie ich rola nie jest kluczowa, jednak jeżeli ktoś nas spyta na ile pewni jesteśmy naszego wyniku odpowiedź brzmi: na 0%. Czy jest zatem sposób byśmy byli czegokolwiek choć trochę pewni?

Tu z pomocą przychodzi nam estymacja przedziałowa, gdzie odpowiedzią będzie nie jeden punkt, ale cały przedział możliwych wartości gdzie dany parametr populacji może się znajdować. Jak wiadomo prawdopodobieństwo trafienia w przedział przy zmiennej losowej ciągłej może już być niezerowe, więc możemy się pozbyć w ten sposób kompromitującej odpowiedzi 0%. Co więcej, tak naprawdę będziemy mieli w tym przypadku kontrolę nad tym z jakim prawdopodobieństwem szukany parametr jest we wskazanym przez nas przedziale, czyli na ile można naszej estymacji ufać.

Zaczniemy od najprostszego przypadku, czyli estymacji przedziałowej wartości średniej przy znanej wariancji dla danej populacji. Znajomość wariancji wcale nie jest taka dziwna w tym przypadku, często może być wynikiem poprzednich badań na podobnych populacjach. Jak już wspomniałem estymacją punktową przy wartości średniej jest po prostu wartość średnia z próbki. Jak w takim razie przerobić ten punkt na przedział? Tu z pomocą przychodzi Centralne Twierdzenie Graniczne, które mówi że rozkład średniej z próbki jest dobrze aproksymowane przez rozkład naturalny. Skoro ma rozkład naturalny, a własności tego rozkładu znamy, to możemy wyznaczyć przedział w którym odpowiedź znajduje się z wybranym prawdopodobieństwem. Rzecz jasna rozkład normalny ma wszędzie niezerową gęstość, więc gdybyśmy chcieli mieć przedział w którym szukana wartość znajduje się na 100% musielibyśmy podać cały zbiór wszystkich liczb rzeczywistych, co byłoby odpowiedzią poprawną, lecz znów niezbyt przydatną. Dlatego też wyznaczamy tak zwany poziom ufności, czyli prawdopodobieństwo z jakim chcemy mieć rację, oznaczane przez $1-\alpha$, gdzie $\alpha$ jest wartością z jaką pozwalamy sobie na błąd. Typowo jest to 5%, ale zdarza się też wartość 2%, 1%, czy 0,1%. Oczywiście przedziałów w których wartość się pojawi na, na przykład, 95% jest nieskończenie wiele, więc z powodów które były podane na wykładzie, wybieramy ten przedział który w środku ma wartość średnią naszej próby. Tym samym przedział ufności dla zadanego poziomu ufności wyznaczamy jako

$ { \LARGE [} \bar{x} - z_{1-\alpha /2}\frac{\sigma}{\sqrt{n}} , \bar{x} + z_{1-\alpha /2}\frac{\sigma}{\sqrt{n}} { \LARGE ]},$

gdzie $\bar{x}$ jest średnią z próbki, $\sigma$ jest odchyleniem standardowym w populacji (czyli pierwiastkiem z wariancji), $n$ jest liczbą elementów w próbce, zaś $z_{1-\alpha /2}$ jest kwantylem rzędu $1-\alpha /2$ standardowego rozkładu normalnego (jeżeli przestraszyliśmy się trochę tej nazwy to technicznie jest to wartość funkcji norm.ppf w punkcie podanym w indeksie(jest to też po prostu w pewien sposób funkcja odwrotna do dystrybuanty)). W tak zdefiniowany przedział można łatwo wyliczyć, że nasza szukana wartość średniej całej populacji wpada z prawdopodobieństwem właśnie $1-\alpha$.

Mając to wszystko na uwadze następna funkcja ma nam właśnie zwrócić dolną i górną granicę przedziału ufności dla danej próbki, znanego odchylenia standardowego i wybranego poziomu popełnienia błędu $\alpha$.

In [ ]:
def esty_sr(lista, od_st , alfa): #lista to wyniki z próby, od_st to odchylenie standardowe, alfa to współczynnik ufności
    n = len(lista)
    x_bar = sum(lista) / n  
    r = norm.ppf(1-alfa/2) * od_st / math.sqrt(n)
    return [float(x_bar - r), float(x_bar+ r)]
    # dla przykładu esty_sr([2,2.3,2.4,2.5,1.9,2.3,2.5,2.1,2.4,2.3], 1 , 0.05) powinnien zwracać [1.6502049676954385, 2.8897950323045616]
print('result:', esty_sr([2,2.3,2.4,2.5,1.9,2.3,2.5,2.1,2.4,2.3], 1 , 0.05))
print('answer:', [1.6502049676954385, 2.8897950323045616])

result: [1.6502049676954385, 2.8897950323045616]
answer: [1.6502049676954385, 2.8897950323045616]


No dobrze, wiemy jak liczyć przedział ufności kiedy znamy wariancję całej populacji, co jednak zrobić jeżeli jej nie znamy? Tu właśnie przydaje nam się nieobciążony estymator wariancji, który możemy potraktować jako przybliżenie wariancji orginalnej populacji. Tym samym wydawać by się mogło, że wystarczy wziąć przedział $$ {\LARGE [} \bar{x} - z_{1-\alpha /2}\frac{s}{\sqrt{n}} , \bar{x} + z_{1-\alpha /2}\frac{s}{\sqrt{n}} {\LARGE ]},$$ gdzie $\bar{x}$ jest średnią z próbki, $s$  pierwiastkiem z nieobciążonego estymatora wariancji (czyli pewnym estymatorem odchylenia standardowego), $n$ jest liczbą elementów w próbce, zaś $z_{1-\alpha /2}$ jest kwantylem rzędu $1-\alpha /2$ standardowego rozkładu normalnego (ciągle liczymy go korzystając z norm.ppf w punkcie podanym w indeksie). I to jest prawda, ale dopiero dla odpowiednio dużego $n$ (zasadniczo $n$ musi być większe od 30). Problem w tym że chociaż estymator wariancji jest nieobciążony i średnio daje wariancję całej populacji, to już jego pierwiastek jest obciążonym estymatorem odchylenia standardowego i średnio będzie się różnił od odchylenia standardowego całej populacji, co będzie widoczne zwłaszcza w małych próbkach. Dlatego też dla próbek o rozmiarze mniejszym bądź równym 30 lepiej jest używać tak zwanego rozkładu $t$, znanego również pod nazwą rozkład Studenta, o $n-1$ stopniach swobody. Wzór na gęstość tego rozkładu jest dosyć skomplikowany, z aktualnego punktu widzenia wystarczy nam wiedzieć, że dla małych próbek wyznaczamy przedział ufności dla wartości średniej wzorami $$ {\LARGE [} \bar{x} - t_{1-\alpha /2, n-1}\frac{s}{\sqrt{n}} , \bar{x} + t_{1-\alpha /2, n-1}\frac{s}{\sqrt{n}} {\LARGE ]},$$ gdzie $\bar{x}$ jest średnią z próbki, $s$  pierwiastkiem z nieobciążonego estymatora wariancji, $n$ jest liczbą elementów w próbce, zaś $t_{1-\alpha /2,n-1}$ jest kwantylem rzędu $1-\alpha /2$ rozkładu t o $n-1$ stopniach swobody (to znowu może wyglądać strasznie jednak wystarczy nam wiedzieć, że liczymy go przy użyciu funkcji t.ppf gdzie dwoma wstawionymi parametrami są $1-\alpha /2$, oraz $n-1$).

Mając to wszystko na uwadze następna funkcja (ta jest dodatkowa) ma, w zależności od rozmiaru próbki, wybrać odpowiedni rozkład i zwrócić dolną i górną wartość przedziału ufności przy zadanym $\alpha$.

In [ ]:
def esty_sr_bez_od(lista , alfa): #lista to wyniki z próby, alfa to współczynnik ufności #dod
    n = len(lista)
    x_bar = sum(lista) / n
    s = math.sqrt(NEW(lista))

    if n <= 30:
        dist = t.ppf(1 - alfa/2, n - 1)
    else:
        dist = norm.ppf(1 - alfa/2)

    r = dist * s / math.sqrt(n)
    
    return [float(x_bar - r), float(x_bar + r)]
print('result:', esty_sr_bez_od([2,2.3,2.4,2.5,1.9,2.3,2.5,2.1,2.4,2.3], 0.05))
print('answer:', [2.1228148457808524, 2.4171851542191476])
print()
print('result:', esty_sr_bez_od([2,2.3,2.4,2.5,1.9,2.3,2.5,2.1,2.4,2.3,2.1,2.5,2.8,2.1,3.0,1.8,2.5, 2.3,2.1,2.7,2.4,2.3,2.1,2.7,2.0,2.1,1.9,1.8,1.6,2.3,2.4,2.1,2.2,2.8,2.1,2.6,3.0,1.7,2.6,2.4], 0.05))
print('answer:', [2.1887914223434586, 2.3962085776565405])
#powinien zwracać [2.1887914223434586, 2.3962085776565405]
#[2.1424767641789475, 2.3975232358210525] jest błędne dla pierwszego
#[2.1854723203337083, 2.3995276796662908] jest błędne dla drugiego

result: [2.12281484577713, 2.41718515422287]
answer: [2.1228148457808524, 2.4171851542191476]

result: [2.188791422343459, 2.396208577656541]
answer: [2.1887914223434586, 2.3962085776565405]


Potrafimy poradzić sobie z przedziałami ufności dla wartości średniej, teraz pora zobaczyć jak one wyglądają dla wariancji. Przybliżanie przez rozkład normalny może wydawać się kuszące, jednak ma poważny problem: jeżeli wariancja jest stosunkowo mała nasz przedział ufności prawie na pewno będzie zawierał wartości ujemne, co dla wariancji w oczywisty sposób nie ma sensu. Jedna okazuje się, że jest inny rozkład który już dobrze opisuje w granicy rozkład wariancji i nazywamy go rozkładem $\chi^2$ o $n-1$ stopniach swobody, gdzie $n$ jest liczbą elementów w próbce. Tu znów nie będziemy wchodzić w szczegóły techniczne, wystarczy wiedzieć, że korzystając z tego rozkładu możemy wyznaczyć przedział ufności wzorem $${\huge [} \frac{(n-1) s^2}{\chi^2_{1-\alpha/2,n-1}},\frac{(n-1) s^2}{\chi^2_{\alpha/2,n-1}} {\huge ]},$$ gdzie $n$ to rozmiar próbki, $s^2$ to nieobciążony estymator wariancji próbki, zaś $\chi^2_{k,n-1}$ jest kwantylem rzędu $k$ rozkładu $\chi^2$ o $n-1$ stopniach swobody (który liczymy korzystając z funkcji chi2.ppf dla parametrów $k$ i $n-1$ ). Zauważmy że w tym przypadku na obu końcach przedziału wstawiamy różne wartości pod $k$.

Mając to wszystko na uwadzę proszę teraz stworzyć funkcję która wyznacza przedział ufności wariancji dla ustalonego $\alpha$ i podaje jego górną i dolną granicę.

In [33]:
def esty_war(lista, alfa): #lista to wyniki z próby, alfa to współczynnik ufności
    n = len(lista)
    numerator = (n - 1) * NEW(lista)
    return [float(numerator / chi2.ppf(1-alfa/2, n-1)), float(numerator/chi2.ppf(alfa/2, n-1))]
    #esty_war([2,2.3,2.4,2.5,1.9,2.3,2.5,2.1,2.4,2.3], 0.05) powinnien zwracać [0.020028631166238927, 0.14109075746397742]
print('result:', esty_war([2,2.3,2.4,2.5,1.9,2.3,2.5,2.1,2.4,2.3], 0.05))
print('answer:', [0.020028631166238927, 0.14109075746397742])

result: [0.020028631166238924, 0.1410907574639774]
answer: [0.020028631166238927, 0.14109075746397742]


Wszystkie liczone do tej pory przedziały dotyczyły zmiennych ilościowych, pytaniem jest czy możemy estymować parametry zmiennych jakościowych. Oczywiście odpowiedź brzmi tak. Najprostszym parametrem jaki możemy badać jest tak zwana proporcja, czyli częstotliwość występowania danej odpowiedzi w populacji. Dla przykładu jeżeli całą naszą populacją jest ulubiony kolor w grupie 4 osób i uzyskane odpowiedzi to: (czerwony, czerwony, niebieski, zielony), to proporcja występowania odpowiedzi "czerwony" wynosi 0,5, zaś odpowiedzi "niebieski" 0,25, tak samo jak "zielony". Okazuje się że najlepszym estymatorem punktowym proporcji dla danej odpowiedzi w próbce jest częstość jego występowania podzielona przez liczbę elementów w próbce (taka swoista wartość średnia). Na podstawie tego faktu, oraz tego że pytanie o częstotliwość to tak naprawdę pytanie o prawdopodobieństwo sukcesu w pewnej próbie Bernoulliego, możemy wyznaczyć przedział ufności dla proporcji wzorami $${\huge [} \hat{p} - z_{1-\alpha /2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}} , \hat{p} + z_{1-\alpha /2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}} {\huge ]},$$ gdzie $\hat{p}$ to proporcja wyznaczona z próbki, $n$ rozmiar próbki, zaś $z_{1-\alpha /2}$ jest kwantylem rzędu $1-\alpha /2$ standardowego rozkładu normalnego (to już powinniśmy rozpoznać jak się liczy). Możemy łatwo zauważyć, że przedział ufności dla proporcji to nic innego jak przedział ufności dla średniej z pewnej próby Bernoulliego o prawdopodobieństwie sukcesu wyznaczonym przez $\hat{p}$.

Mając to wszystko na uwadze proszę na koniec stworzyć funkcję (ta również jest dodatkowa) która wyznacza przedział ufności dla proporcji gdzie podajemy próbkę jakościową, pewną wybraną wartość dla której chcemy tę proporcję badać, oraz wyznaczonego $\alpha$. Funkcja ma oddać dolną i górną granicę tego przedziału.

In [52]:
def esty_prop(lista, wart, alfa):#lista to wyniki z próby, wart to wartość której proporcję estymujemy, alfa to współczynnik ufności #dod
    n = len(lista)
    p_hat = lista.count(wart) / n
    r = norm.ppf(1-alfa/2) * math.sqrt(p_hat * (1 - p_hat) / n)
    return [float(p_hat - r), float(p_hat + r)]
    #esty_prop(["niebieski","zielony","czerwony","żółty", "czerwony", "czerwony","zielony","zielony","żółty" ,"czerwony","zielony","żółty","niebieski"] , "czerwony", 0.05)
  #powinien zwracać [0.056801752272596207, 0.5585828631120192]
print('result:', esty_prop(["niebieski","zielony","czerwony","żółty", "czerwony", "czerwony","zielony","zielony","żółty" ,"czerwony","zielony","żółty","niebieski"], "czerwony", 0.05))
print('answer:', [0.056801752272596207, 0.5585828631120192])

result: [0.056801752272596207, 0.5585828631120192]
answer: [0.056801752272596207, 0.5585828631120192]


Jako zadania dodatkowe proszę spróbować zastosować odpowiednią funkcję w następujących problemach:

W losowej próbce rodzin zapytano jak często w tygodniu wychodzą na zakupy. Odpowiedzi zebrano w poniższej liście:

ZAKUPY = [2,2,2,1,4,2,3,2,5,4,2,3,5,0,3,2,3,1,4,3,3,2,1,6,2]

Wyznacz przedział ufności dla średniej liczby wyjść na zakupy o poziomie ufności 95%.

In [39]:
print(esty_sr_bez_od([2,2,2,1,4,2,3,2,5,4,2,3,5,0,3,2,3,1,4,3,3,2,1,6,2], 0.05))

[2.0996565818178223, 3.260343418182178]


W pewnym przeszkolu zapytano dzieci o ulubiony kolor i zebrane wyniki przedstawiono w tabeli KOLORY. Czy powiedzenie że jedna trzecia dzieci lubi kolor czerwony byłoby zgodne informacją z przedziału ufności dla poziomu ufności 95%?

KOLORY=["fioletowy", "żółty"  ,   "różowy"   , "zielony"  , "różowy" , "fioletowy" ,"czerwony" , "fioletowy" ,"żółty"  ,   "żółty" ,    "różowy" ,   "niebieski" ,"czerwony",  "różowy",    "zielony" ,  "niebieski", "fioletowy", "czerwony" , "zielony",  "czerwony",  "różowy",   "czerwony" , "niebieski", "zielony"  , "różowy" ,   "zielony"  , "czerwony"  ,"fioletowy", "czerwony",  "fioletowy" ,"różowy"   , "różowy",    "zielony" ,  "różowy"  ,  "różowy" ,
"zielony"  , "fioletowy", "żółty"  ,   "różowy"  ,  "fioletowy" ,"czerwony"  ,"żółty",     "czerwony" , "niebieski", "różowy" ,   "niebieski", "zielony" ,  "różowy" ,   "fioletowy", "czerwony" ]

In [ ]:
KOLORY=["fioletowy", "żółty"  ,   "różowy"   , "zielony"  , "różowy" , "fioletowy" ,"czerwony" , "fioletowy" ,"żółty"  ,   "żółty" ,    "różowy" ,   "niebieski" ,"czerwony",  "różowy",    "zielony" ,  "niebieski", "fioletowy", "czerwony" , "zielony",  "czerwony",  "różowy",   "czerwony" , "niebieski", "zielony"  , "różowy" ,   "zielony"  , "czerwony"  ,"fioletowy", "czerwony",  "fioletowy" ,"różowy"   , "różowy",    "zielony" ,  "różowy"  ,  "różowy" , "zielony"  , "fioletowy", "żółty"  ,   "różowy"  ,  "fioletowy" ,"czerwony"  ,"żółty",     "czerwony" , "niebieski", "różowy" ,   "niebieski", "zielony" ,  "różowy" ,   "fioletowy", "czerwony" ]

result = esty_prop(KOLORY, 'czerwony', 0.05)
print(result)
if result[0] <= 1/3 <= result[1]:
    print('TAK')
else:
    print('NIE')

[0.08912769405202577, 0.31087230594797427]
NIE



Liczba godzin spędzonych dziennie przed telewizorem została zmierzona na losowej próbce rodzin. Wyniki zostały podane na następującej liście:

TV=[3.7, 4.2, 1.5, 3.6, 5.9, 4.7, 8.2, 3.9, 2.5, 4.4, 2.1, 3.6, 1.1, 7.3, 4.2, 3.0, 3.8, 2.2, 4.2, 3.8, 4.3, 2.1, 2.4, 6.0, 3.7, 2.5, 1.3, 2.8, 3.0, 5.6]

Wyznacz przedział ufności dla wartości średniej godzin spędzonych dziennie przed telewizorem o poziomie ufności 99.8%.

In [43]:
TV=[3.7, 4.2, 1.5, 3.6, 5.9, 4.7, 8.2, 3.9, 2.5, 4.4, 2.1, 3.6, 1.1, 7.3, 4.2, 3.0, 3.8, 2.2, 4.2, 3.8, 4.3, 2.1, 2.4, 6.0, 3.7, 2.5, 1.3, 2.8, 3.0, 5.6]
esty_sr_bez_od(TV, 0.002)

[2.6855796020399434, 4.754420397960056]

W korporacji postanowiono zmierzyć średni czas nieproduktywnego przeglądania internetu przez pracowników. W poprzednich takich badaniach odchylenie standardowe wynosiło 8.2 minuty. Wyniki zebrano w poniższej liście:

NET=[31.48, 32.29, 34.78, 19.24, 33.90, 25.46, 8.65, 38.55, 17.51, 26.41, 24.25, 34.75, 15.84, 37.84, 20.66, 18.82, 19.33, 20.94, 30.89, 29.41, 37.48, 20.53, 19.72, 24.89, 17.14, 22.36, 31.74, 41.66, 33.30, 38.15, 28.42, 33.74, 31.22, 37.04, 29.81, 26.23, 26.92, 45.45, 24.02, 16.47, 14.96, 21.55, 25.85, 18.07, 25.87, 52.45, 16.28, 32.93, 32.82, 41.93]

Wyznacz przedział ufności dla wrtości średniej o poziomie ufności 99.5%.


In [45]:
NET=[31.48, 32.29, 34.78, 19.24, 33.90, 25.46, 8.65, 38.55, 17.51, 26.41, 24.25, 34.75, 15.84, 37.84, 20.66, 18.82, 19.33, 20.94, 30.89, 29.41, 37.48, 20.53, 19.72, 24.89, 17.14, 22.36, 31.74, 41.66, 33.30, 38.15, 28.42, 33.74, 31.22, 37.04, 29.81, 26.23, 26.92, 45.45, 24.02, 16.47, 14.96, 21.55, 25.85, 18.07, 25.87, 52.45, 16.28, 32.93, 32.82, 41.93]
print(esty_sr(NET, 8.2, 0.005))

[24.54480891531052, 31.055191084689483]


Fabryka produkuje stalowe pręty. Pręty te mają mieć średnią średnicę rzędu 30 mm. W ramach przygotowań to testów jakościowych postanowiono wyznaczyć estymator wariancji i wylosowano 10 próbnych prętów. Poniższa lista zawiera ich średnice mierzone w milimetrach:

SREDNIE=[34, 32, 35, 33, 30, 29, 31, 32, 28, 32]

Wyznacz estymator wariancji, oraz przedział ufności wariancji na poziomie 95% na podstawie tej próbki.

In [49]:
SREDNIE=[34, 32, 35, 33, 30, 29, 31, 32, 28, 32]
e_wariancji = NEW(SREDNIE)
przedzial = esty_war(SREDNIE, 0.05)
print(e_wariancji)
print(przedzial)

4.711111111111111
[2.22890803529798, 15.701438625912447]
